# Backpropagation from Scratch: Spiral Classification

ตัวอย่างนี้สร้าง Neural Network ด้วย **NumPy ล้วน ๆ** เพื่อจำแนกข้อมูลรูปเกลียว 3 คลาส ซึ่งเส้นตรงธรรมดาแบ่งไม่ได้

สิ่งที่จะเห็น: `Forward pass → Loss → Backpropagation → Update parameters` พร้อม Decision Boundary และ Gradient Checking

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
print('NumPy:', np.__version__)

## 1) สร้างข้อมูลรูปเกลียว 3 คลาส
แต่ละจุดมี input 2 ค่า `(x1, x2)` และ target เป็นคลาส `0, 1, 2`

In [ ]:
def make_spiral(points_per_class=150, classes=3, noise=0.20):
    X = np.zeros((points_per_class * classes, 2))
    y = np.zeros(points_per_class * classes, dtype=int)
    for c in range(classes):
        ix = slice(c * points_per_class, (c + 1) * points_per_class)
        radius = np.linspace(0.05, 1, points_per_class)
        theta = np.linspace(c * 4, (c + 1) * 4, points_per_class)
        theta += np.random.randn(points_per_class) * noise
        X[ix] = np.c_[radius * np.sin(theta), radius * np.cos(theta)]
        y[ix] = c
    order = np.random.permutation(len(X))
    return X[order], y[order]

X, y = make_spiral()
colors = ListedColormap(['#ef476f', '#06d6a0', '#118ab2'])
plt.scatter(X[:, 0], X[:, 1], c=y, cmap=colors, s=24, edgecolor='white', linewidth=0.25)
plt.title('Spiral dataset: a nonlinear problem')
plt.xlabel('x1'); plt.ylabel('x2'); plt.axis('equal'); plt.show()
print('X shape:', X.shape, '| y shape:', y.shape)

## 2) โครงสร้าง Neural Network

`Input(2) → Dense(32) → tanh → Dense(3) → softmax`

Forward pass:
- $Z_1=XW_1+b_1$
- $A_1=\tanh(Z_1)$
- $Z_2=A_1W_2+b_2$
- $\hat{Y}=\mathrm{softmax}(Z_2)$

In [ ]:
def initialize(input_size=2, hidden_size=32, output_size=3, seed=7):
    rng = np.random.default_rng(seed)
    return {
        'W1': rng.normal(0, np.sqrt(1/input_size), (input_size, hidden_size)),
        'b1': np.zeros((1, hidden_size)),
        'W2': rng.normal(0, np.sqrt(1/hidden_size), (hidden_size, output_size)),
        'b2': np.zeros((1, output_size))
    }

def softmax(z):
    shifted = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(shifted)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def forward(X, p):
    Z1 = X @ p['W1'] + p['b1']
    A1 = np.tanh(Z1)
    Z2 = A1 @ p['W2'] + p['b2']
    probs = softmax(Z2)
    return probs, {'X': X, 'Z1': Z1, 'A1': A1, 'Z2': Z2, 'probs': probs}

def cross_entropy(probs, y):
    return -np.mean(np.log(probs[np.arange(len(y)), y] + 1e-12))

params = initialize()
probs_before, _ = forward(X, params)
print('Initial loss:', round(cross_entropy(probs_before, y), 4))
print('First prediction probabilities:', np.round(probs_before[0], 3))

## 3) Backpropagation หัวใจของการเรียนรู้
Backpropagation ส่งค่าความผิดพลาดย้อนจาก output กลับไปหาแต่ละ weight ด้วย Chain Rule

สำหรับ Softmax + Cross-Entropy จะได้อนุพันธ์ที่สวยมาก: $dZ_2=(\hat{Y}-Y)/N$

In [ ]:
def backward(y, p, cache):
    n = len(y)
    dZ2 = cache['probs'].copy()
    dZ2[np.arange(n), y] -= 1
    dZ2 /= n

    dW2 = cache['A1'].T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)
    dA1 = dZ2 @ p['W2'].T
    dZ1 = dA1 * (1 - cache['A1'] ** 2)  # derivative of tanh
    dW1 = cache['X'].T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)
    return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

def accuracy(X, y, p):
    probs, _ = forward(X, p)
    return np.mean(np.argmax(probs, axis=1) == y)

_, cache = forward(X, params)
grads = backward(y, params, cache)
for name, grad in grads.items():
    print(f'd{name}: shape={grad.shape}, norm={np.linalg.norm(grad):.5f}')

## 4) Gradient Checking
เปรียบเทียบ gradient จาก Backpropagation กับค่าประมาณเชิงตัวเลข หากใกล้กันมาก แสดงว่าสูตรและโค้ดมีแนวโน้มถูกต้อง

In [ ]:
def gradient_check(X, y, p, parameter='W1', index=(0, 0), eps=1e-5):
    probs, cache = forward(X, p)
    analytical = backward(y, p, cache)[parameter][index]
    original = p[parameter][index]
    p[parameter][index] = original + eps
    loss_plus = cross_entropy(forward(X, p)[0], y)
    p[parameter][index] = original - eps
    loss_minus = cross_entropy(forward(X, p)[0], y)
    p[parameter][index] = original
    numerical = (loss_plus - loss_minus) / (2 * eps)
    relative_error = abs(analytical - numerical) / (abs(analytical) + abs(numerical) + 1e-12)
    return analytical, numerical, relative_error

a, n, err = gradient_check(X[:30], y[:30], params, 'W1', (0, 0))
print(f'Backprop gradient : {a:.8f}')
print(f'Numerical gradient: {n:.8f}')
print(f'Relative error    : {err:.2e}')
print('PASS' if err < 1e-6 else 'CHECK THE DERIVATIVE')

## 5) Train: ปรับ Weight ย้อนทิศทางของ Gradient
$\theta \leftarrow \theta - \eta\nabla_\theta L$ โดย $\eta$ คือ learning rate

In [ ]:
def train(X, y, hidden_size=32, learning_rate=1.0, epochs=3000):
    p = initialize(hidden_size=hidden_size)
    history = {'loss': [], 'accuracy': []}
    snapshots = {}
    for epoch in range(epochs + 1):
        probs, cache = forward(X, p)
        loss = cross_entropy(probs, y)
        acc = np.mean(np.argmax(probs, axis=1) == y)
        history['loss'].append(loss)
        history['accuracy'].append(acc)
        if epoch in [0, 50, 300, 1000, epochs]:
            snapshots[epoch] = {k: v.copy() for k, v in p.items()}
        if epoch == epochs:
            break
        grads = backward(y, p, cache)
        for key in p:
            p[key] -= learning_rate * grads[key]
        if epoch % 500 == 0:
            print(f'Epoch {epoch:4d} | loss={loss:.4f} | accuracy={acc:.1%}')
    return p, history, snapshots

params, history, snapshots = train(X, y)
print(f'Final accuracy: {accuracy(X, y, params):.2%}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history['loss'], color='#ef476f', linewidth=2)
ax[0].set(title='Loss decreases', xlabel='Epoch', ylabel='Cross-entropy loss')
ax[1].plot(np.array(history['accuracy']) * 100, color='#118ab2', linewidth=2)
ax[1].set(title='Accuracy increases', xlabel='Epoch', ylabel='Accuracy (%)', ylim=(25, 102))
plt.tight_layout(); plt.show()

## 6) ดูสมองของโมเดลค่อย ๆ เรียนรู้
สีพื้นหลังคือคลาสที่โมเดลทำนาย ส่วนจุดคือคำตอบจริง

In [ ]:
def plot_boundary(ax, X, y, p, title):
    pad = 0.15
    x_min, x_max = X[:, 0].min()-pad, X[:, 0].max()+pad
    y_min, y_max = X[:, 1].min()-pad, X[:, 1].max()+pad
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
    grid = np.c_[xx.ravel(), yy.ravel()]
    pred = np.argmax(forward(grid, p)[0], axis=1).reshape(xx.shape)
    ax.contourf(xx, yy, pred, cmap=colors, alpha=0.28)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=colors, s=13, edgecolor='white', linewidth=0.2)
    ax.set_title(title); ax.set_aspect('equal'); ax.grid(False)

epochs_to_show = [0, 50, 300, 1000, 3000]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.6))
for ax, epoch in zip(axes, epochs_to_show):
    acc = accuracy(X, y, snapshots[epoch])
    plot_boundary(ax, X, y, snapshots[epoch], f'Epoch {epoch}\nAccuracy {acc:.1%}')
plt.tight_layout(); plt.show()

## 7) ทดลองปรับเอง
ลองเปลี่ยน `hidden_size`, `learning_rate` หรือ `epochs` แล้วสังเกตผล

คำถามชวนคิด:
1. Hidden neurons น้อยเกินไปเกิดอะไรขึ้น?
2. Learning rate สูงหรือต่ำเกินไปส่งผลอย่างไร?
3. ถ้าตัด activation `tanh` ออก โมเดลยังแบ่งเกลียวได้หรือไม่?

In [ ]:
# Playground: แก้ตัวเลขแล้ว Run cell นี้ใหม่
my_hidden_size = 16
my_learning_rate = 1.0
my_epochs = 2000

my_params, my_history, _ = train(
    X, y, hidden_size=my_hidden_size,
    learning_rate=my_learning_rate, epochs=my_epochs
)
fig, ax = plt.subplots(figsize=(6, 5))
plot_boundary(ax, X, y, my_params,
              f'hidden={my_hidden_size}, lr={my_learning_rate}, epochs={my_epochs}\n'
              f'accuracy={accuracy(X, y, my_params):.1%}')
plt.show()

## สรุป Flow ของ Backpropagation

1. **Forward:** คำนวณค่าทำนายจาก input ไป output
2. **Loss:** วัดว่าค่าทำนายต่างจาก target เท่าใด
3. **Backward:** ใช้ Chain Rule หา gradient ของทุก weight และ bias
4. **Update:** ขยับ parameters ไปทิศที่ทำให้ loss ลดลง
5. ทำซ้ำหลาย epoch จน decision boundary เหมาะกับข้อมูล